# Environment and inference smoke test

Original learning notebook, using local project folders and CUDA when available.


In [ ]:
%pip install -q "ultralytics==8.4.115"


In [ ]:
# Runtime and GPU check

In [ ]:
import platform
import torch

print(platform.python_version())
print(torch.__version__)
print(torch.cuda.is_available())

In [ ]:
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    properties = torch.cuda.get_device_properties(0)
    free_bytes, total_bytes = torch.cuda.mem_get_info(0)
    print(gpu_name)
    print(torch.cuda.get_device_capability(0))
    print(f"Total: {total_bytes / 1024**3:.2f} GiB")
    print(f"Allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GiB")
    print(f"Reserved: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GiB")
else:
    gpu_name = "CPU"
    print("CUDA unavailable; using CPU for the smoke test.")
DEVICE = 0 if torch.cuda.is_available() else "cpu"


In [ ]:
# Keep generated files in the current working directory.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd() / "aquarium-experiment"
DATA_DIR = PROJECT_ROOT / "data"
RUNS_DIR = PROJECT_ROOT / "runs"
WEIGHTS_DIR = PROJECT_ROOT / "weights"
REPORTS_DIR = PROJECT_ROOT / "reports"


In [ ]:
for directory in [
    PROJECT_ROOT,
    DATA_DIR,
    WEIGHTS_DIR,
    REPORTS_DIR
]: directory.mkdir(parents = True, exist_ok = True)


print("Project root:", PROJECT_ROOT)
print("Data:", DATA_DIR)
print("Runs:", RUNS_DIR)
print("Weights:", WEIGHTS_DIR)
print("Reports:", REPORTS_DIR)

Smoke Test

In [ ]:
import torch

print(torch.cuda.is_available())

In [ ]:
import json
from ultralytics import YOLO

MODEL_PATH = WEIGHTS_DIR / "yolo11n.pt"
SMOKE_DIR = REPORTS_DIR / "smoke"
OUTPUT_PATH = SMOKE_DIR / "bus_yolo11n_gpu.jpg"
MANIFEST_PATH = SMOKE_DIR / "smoke_manifest.json"

SAMPLE_URL = "https://ultralytics.com/images/bus.jpg"

WEIGHTS_DIR.mkdir(parents = True, exist_ok = True)
SMOKE_DIR.mkdir(parents = True, exist_ok = True)

model = YOLO(str(MODEL_PATH))

result = model.predict(
    source = SAMPLE_URL,
    device = DEVICE,
    imgsz = 640,
    verbose = True
)

# organize result:

inference_device = result[0].boxes.data.device
result[0].save(filename = str(OUTPUT_PATH))
manifest = {
    "model": "yolo11n.pt",
    "model_path": str(MODEL_PATH),
    "source": SAMPLE_URL,
    "gpu": gpu_name,
    "inference_device": str(inference_device),
    "detections": len(result[0].boxes),
    "output_path": str(OUTPUT_PATH),
}

MANIFEST_PATH.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(json.dumps(manifest, ensure_ascii=False, indent=2))
print("Prediction saved:", OUTPUT_PATH)